In [1]:
import json
import glob
import pandas as pd

In [2]:
# Gallery
artworks_dict = json.load(open("exports/production/artworks.json", "r"))
categories_dict = json.load(open("exports/production/categories.json", "r"))
techniques_dict = json.load(open("exports/production/techniques.json", "r"))

# Pages
pages_dict = json.load(open("exports/production/pages.json", "r"))
page_attachments_dict = json.load(open("exports/production/page_attachments.json", "r"))
images_dict = json.load(open("exports/production/images.json", "r"))

# Studies
studies_dict = json.load(open("exports/production/studies.json", "r"))
image_attachments_dict = json.load(open("exports/production/image_attachments.json", "r"))
attachments_dict = json.load(open("exports/production/attachments.json", "r"))
images_dict = json.load(open("exports/production/images.json", "r"))

# Intro slideshows
intro_slideshows_dict = json.load(open("exports/production/intro_slideshows.json", "r"))
slides_dict = json.load(open("exports/production/slides.json", "r"))


In [24]:
gallery0 = (
    pd.DataFrame(artworks_dict)[["id", "title", "date_created", "xinch", "yinch", "xcm", "ycm", "filename", "category_id", "technique_id"]]
    .merge(pd.DataFrame(categories_dict)[["id", "name"]].rename(columns={"name": "category", "id": "category_id"}), on="category_id")
    .merge(pd.DataFrame(techniques_dict)[["id", "name"]].rename(columns={"name": "technique", "id": "technique_id"}), on="technique_id")
    .drop(columns=["category_id", "technique_id"])
)
print(gallery0.shape)
gallery0.head(2)


(677, 10)


,id,title,date_created,xinch,yinch,xcm,ycm,filename,category,technique
0,1474,Victory,1/1/2002,24.0,16.0,60.0,40.0,1.jpg,Print,Engraving
1,1475,Dove 3,1/1/2002,16.0,28.0,40.0,70.0,2.jpg,Print,Engraving


In [26]:
gallery0.to_json("data/gallery.json", orient="records")

In [27]:
images_for_studies = (
    pd.DataFrame(image_attachments_dict)
    [['study_id', 'image_id']]
    .groupby('study_id')
    ['image_id']
    .agg(list)
    .to_dict()
)
json.dump(images_for_studies, open("data/images_for_studies.json", "w"))
artworks_for_studies = (
    pd.DataFrame(attachments_dict)
    [['study_id', 'artwork_id']]
    .groupby('study_id')
    ['artwork_id']
    .agg(list)
    .to_dict()
)
json.dump(artworks_for_studies, open("data/artworks_for_studies.json", "w"))
artworks_filenames_and_titles = (
    gallery0
    [['id', 'filename', 'title']]
    .set_index('id')
    .to_dict(orient='index')
)
json.dump(artworks_filenames_and_titles, open("data/artworks_filenames_and_titles.json", "w"))
images_filenames = ( 
    pd.DataFrame(images_dict)
    [['id', 'filename']]
    .set_index('id')
    ['filename']
    .to_dict()
)
json.dump(images_filenames, open("data/images_filenames.json", "w"))

In [23]:
artworks_template = """
    <img src="gallery/{filename}" alt="{title}" />
"""
images_template = """
    <img src="images/{filename}" />
"""

body_template = """
    <h1>{name}</h1>
    {description}
    {artworks}
    {images}
"""

for _, row in pd.DataFrame(studies_dict).iterrows():
    id = row['id']
    slug = row['name'].lower().replace(" ", "-")
    name = row['name']
    description = row['description']
    images_data = images_for_studies.get(id, [])
    artworks_data = artworks_for_studies.get(id, [])

    images = "".join([images_template.format(filename=images_filenames[image_id]) for image_id in images_data])
    artworks = "".join([artworks_template.format(filename=artworks_filenames_and_titles[artwork_id]['filename'], title=artworks_filenames_and_titles[artwork_id]['title']) for artwork_id in artworks_data])

    body = body_template.format(name=name, description=description, artworks=artworks, images=images)

    with open(f"studies/{slug}.html", "w") as f:
        f.write(body)
